```{contents}
```
## Early Stopping

**Early stopping** is a regularization and training control technique that halts training when the model’s performance on a **validation set** stops improving.
It prevents:

* **Overfitting**: model continues to fit training noise
* **Wasted computation**: unnecessary epochs after convergence
* **Model degradation**: validation loss increases despite decreasing training loss

Formally, training is stopped when:

$$
\text{val_metric}(t) \ge \min_{k < t} \text{val_metric}(k) \quad \text{for } p \text{ consecutive epochs}
$$

where $p$ is called **patience**.

---

### Intuition

Training behaves like descending into a valley:

* **Training loss** always goes down
* **Validation loss** goes down until the model starts memorizing noise
* After that point, validation loss **rises** even though training loss decreases

Early stopping detects the bottom of the valley and stops there.

```
Loss
│\           validation
│ \__        /
│    \__   /
│        \_/   ← stop here
│
└───────────────── Epochs
```

---

### Training Workflow with Early Stopping

1. Split data into **train / validation**
2. Train model epoch by epoch
3. After each epoch:

   * Evaluate on validation set
   * Track best validation metric
4. If no improvement for `patience` epochs:

   * Stop training
   * Restore best model weights

---

### Algorithm

```python
best_metric = +∞
counter = 0

for epoch in range(max_epochs):
    train()
    val_metric = validate()

    if val_metric < best_metric:
        best_metric = val_metric
        save_model()
        counter = 0
    else:
        counter += 1

    if counter >= patience:
        load_best_model()
        break
```

---

### Why Early Stopping Works

| Property       | Effect                            |
| -------------- | --------------------------------- |
| Regularization | Limits effective model capacity   |
| Generalization | Stops before overfitting          |
| Stability      | Prevents catastrophic degradation |
| Efficiency     | Reduces training time             |

It behaves similarly to **L2 regularization** by limiting parameter growth indirectly.

---

### PyTorch Demonstration

```python
import torch
from torch import nn, optim

class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.best = float("inf")
        self.counter = 0
        self.best_state = None

    def step(self, val_loss, model):
        if val_loss < self.best - self.min_delta:
            self.best = val_loss
            self.counter = 0
            self.best_state = {k: v.clone() for k,v in model.state_dict().items()}
        else:
            self.counter += 1

        if self.counter >= self.patience:
            model.load_state_dict(self.best_state)
            return True
        return False
```

Usage in training loop:

```python
early_stop = EarlyStopping(patience=7)

for epoch in range(100):
    train_loss = train_epoch(model, train_loader)
    val_loss = validate_epoch(model, val_loader)

    if early_stop.step(val_loss, model):
        print(f"Stopped at epoch {epoch}")
        break
```

---

### Key Hyperparameters

| Parameter    | Role                                     |
| ------------ | ---------------------------------------- |
| patience     | how long to wait for improvement         |
| min_delta    | minimum change to qualify as improvement |
| monitor      | validation loss or metric                |
| restore_best | reload best weights after stop           |

---

### Variants and Extensions

| Variant                 | Description                               |
| ----------------------- | ----------------------------------------- |
| Metric-based            | Monitor accuracy, F1, AUC instead of loss |
| Smoothed early stopping | Uses moving average of metric             |
| Multi-objective         | Monitors multiple validation signals      |
| Adaptive patience       | Adjusts patience based on learning rate   |

---

### Failure Modes and Remediation

| Failure              | Cause                                 | Remedy                              |
| -------------------- | ------------------------------------- | ----------------------------------- |
| Stops too early      | Noisy validation                      | Increase patience, smooth metric    |
| Never stops          | Validation keeps improving marginally | Increase `min_delta`                |
| Overfitting persists | Weak validation split                 | Improve data split                  |
| Underfitting         | Stop occurs too early                 | Increase patience or model capacity |

---

### When to Use Early Stopping

| Scenario               | Recommendation       |
| ---------------------- | -------------------- |
| Small datasets         | Strongly recommended |
| Large models           | Mandatory            |
| High variance data     | Highly beneficial    |
| Pretrained fine-tuning | Essential            |

---

### Conceptual Summary

Early stopping transforms the training process into a **controlled optimization** problem where **generalization**, not training loss, defines convergence.
It provides one of the most effective and inexpensive forms of regularization in modern deep learning.
